# Aggregate Metadata — Oddball Response

Reads all `step1b_a-metadata.csv` files from L: and concatenates them
into a single `all_metadata.csv` per paradigm.

**Output on NAS:**
- `{date}_active_{n}p_all_metadata.csv`


In [1]:
from datetime import datetime
from pathlib import Path
import sys
import glob
import pandas as pd

# sys.path.insert(0, str(Path.cwd().parents[1] / "_notebooks_for_publication"))

DATE_STR = datetime.today().strftime("%Y%m%d")

if sys.platform == "win32":
    root_active  = Path("L:/EEG_Flow_data/Study_EEG_Flow_Active")
else:
    root_active  = Path("/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active")

derivatives = root_active / "derivatives_v2"
out_path    = derivatives / "outputs" / "measures" / "behavior"
out_path.mkdir(parents=True, exist_ok=True)

# Group → condition mapping
FLOW_MAP = {
    ("UT",      "1", "G1"): "low_flow",  ("UT", "1", "G2"): "high_flow",
    ("UT",      "1", "G3"): "low_flow",  ("UT", "1", "G4"): "high_flow",
    ("UT",      "1", "G5"): "low_flow",  ("UT", "1", "G6"): "high_flow",
    ("UT",      "1", "G7"): "low_flow",  ("UT", "1", "G8"): "high_flow",
    ("UT",      "2", "G1"): "high_flow", ("UT", "2", "G2"): "low_flow",
    ("UT",      "2", "G3"): "high_flow", ("UT", "2", "G4"): "low_flow",
    ("UT",      "2", "G5"): "high_flow", ("UT", "2", "G6"): "low_flow",
    ("UT",      "2", "G7"): "high_flow", ("UT", "2", "G8"): "low_flow",
    ("oddball", "1", None): "oddball_1",
    ("oddball", "2", None): "oddball_2",
}

def map_condition(task, run, group):
    if task == "oddball":
        return f"oddball_{run}"
    return FLOW_MAP.get((task, run, group), "unknown")

print("Config OK")
print(f"Derivatives : {derivatives}")
print(f"Output      : {out_path}")


Config OK
Derivatives : L:\EEG_Flow_data\Study_EEG_Flow_Active\derivatives_v2
Output      : L:\EEG_Flow_data\Study_EEG_Flow_Active\derivatives_v2\outputs\measures\behavior


In [2]:
# ==============================================================================
# Glob all step1b metadata files
# ==============================================================================

pattern = str(derivatives / "**" / "*step1b_a-metadata.csv")
all_files = [f for f in glob.glob(pattern, recursive=True) if "old" not in f]

print(f"Found {len(all_files)} metadata files")

Found 195 metadata files


In [3]:
# ==============================================================================
# Concatenate all metadata files
# ==============================================================================

rows = []
skipped = []

for file in sorted(all_files):
    path   = Path(file)
    parts  = path.parts

    try:
        # Parse participant, group, task, run from path
        # e.g. derivatives/sub-P01-G1/sub-P01-G1_task-UT_run-1/sub-P01-G1_task-UT_run-1_step1b_a-metadata.csv
        sub_dir   = [p for p in parts if p.startswith("sub-")][0]
        _, uid, gid = sub_dir.split("-")

        fname_stem = path.stem  # sub-P01-G1_task-UT_run-1_step1b_a
        task = fname_stem.split("task-")[1].split("_")[0]
        run  = fname_stem.split("run-")[1].split("_")[0]

        df_tmp = pd.read_csv(file).reset_index()
        df_tmp["participant"] = uid
        df_tmp["group"]       = gid
        df_tmp["task"]        = task
        df_tmp["run"]         = run
        df_tmp["condition"]   = map_condition(task, run, gid)

        # Keep only relevant columns
        cols = ["participant", "group", "task", "run", "condition",
                "event_name", "response", "response_type", "response_correct"]
        df_tmp = df_tmp[[c for c in cols if c in df_tmp.columns]]

        rows.append(df_tmp)

    except Exception as e:
        skipped.append((file, str(e)))
        print(f"  Skipped: {path.name} — {e}")

df_all = pd.concat(rows, ignore_index=True)
n_participants = df_all["participant"].nunique()

print(f"\nTotal rows     : {len(df_all):,}")
print(f"N participants : {n_participants}")
print(f"Skipped        : {len(skipped)}")
print(f"\nTasks          : {sorted(df_all['task'].unique())}")
print(f"Conditions     : {sorted(df_all['condition'].unique())}")
print(f"Response types : {sorted(df_all['response_type'].unique())}")
df_all.head(4)



Total rows     : 132,910
N participants : 49
Skipped        : 0

Tasks          : ['UT', 'oddball']
Conditions     : ['high_flow', 'low_flow', 'oddball_1', 'oddball_2']
Response types : ['CorrectRejections_novel', 'CorrectRejections_standard', 'FalseAlarms_novel', 'FalseAlarms_standard', 'FalseAlarms_tooquick', 'Hits', 'Misses']


,participant,group,task,run,condition,event_name,response,response_type,response_correct
0,P01,G1,UT,1,low_flow,standard,NaN,CorrectRejections_standard,True
1,P01,G1,UT,1,low_flow,standard,NaN,CorrectRejections_standard,True
2,P01,G1,UT,1,low_flow,standard,NaN,CorrectRejections_standard,True
3,P01,G1,UT,1,low_flow,standard,NaN,CorrectRejections_standard,True


In [4]:
# ==============================================================================
# Sanity checks
# ==============================================================================

# Runs per participant
runs_per_p = df_all.groupby("participant")[["task", "run"]].apply(
    lambda x: x.drop_duplicates().shape[0]
).reset_index(name="n_runs")
print("Runs per participant:")
print(runs_per_p["n_runs"].value_counts().to_string())
print()

# Participants with != 4 runs
unexpected = runs_per_p[runs_per_p["n_runs"] != 4]
if len(unexpected) > 0:
    print("WARNING — unexpected run counts:")
    print(unexpected.to_string(index=False))
else:
    print("All participants have 4 runs ✓")
print()

# Response type distribution
print("Response type distribution:")
print(df_all["response_type"].value_counts().to_string())


Runs per participant:
n_runs
4    48
3     1

WARNING — unexpected run counts:
participant  n_runs
        P75       3

Response type distribution:
response_type
CorrectRejections_standard    104988
CorrectRejections_novel        13105
Hits                           10900
Misses                          2371
FalseAlarms_standard            1338
FalseAlarms_novel                187
FalseAlarms_tooquick              21


In [5]:
# ==============================================================================
# Export
# ==============================================================================

fname = out_path / f"{DATE_STR}_active_all_metadata.csv"
df_all.to_csv(fname, index=False)
print(f"Saved -> {fname.name}")


Saved -> 20261016_active_all_metadata.csv
